In [3]:
import numpy as np

import pennylane as qml

In [4]:
device = qml.device("default.qubit", wires=4)

In [5]:
from pennylane.measurements import ExpectationMP

# set qubit state from angle matrix
def init_qubit(matrix: np.ndarray, qubit: int):
    qml.RX(phi=matrix[0], wires=qubit)
    qml.RY(phi=matrix[1], wires=qubit)
    qml.RZ(phi=matrix[2], wires=qubit)

# get qubit state as matrix
def get_qubit_state(qubit: int) -> tuple[ExpectationMP, ExpectationMP, ExpectationMP]:
    return qml.expval(qml.PauliX(wires=qubit)), qml.expval(qml.PauliY(wires=qubit)), qml.expval(qml.PauliZ(wires=qubit))

In [6]:
# from abc import abstractmethod
# from typing import Protocol

# class NativeGates(Protocol):
#     @abstractmethod
#     def CNOT(   )

In [7]:
@qml.qnode(device)
def quantum_teleport(matrix: np.ndarray):
    # apply angle matrix to Alice's qubit
    init_qubit(matrix, 0)

    qml.Hadamard(wires=1)
    qml.CNOT(wires=[1, 2])
    qml.CNOT(wires=[0, 1])
    qml.Hadamard(wires=0)

    # measure 1 and 2 qubits
    m0 = qml.measure(0)  # Z
    m1 = qml.measure(1)  # X
    
    qml.cond(m1, qml.PauliX)(wires=2) # type: ignore
    qml.cond(m0, qml.PauliZ)(wires=2) # type: ignore

    return get_qubit_state(2)


In [8]:
@qml.qnode(device)
def debug(matrix: np.ndarray):
    init_qubit(matrix, 3)

    return get_qubit_state(3)

In [9]:
# задаем матрицу углов поворота (X, Y, Z)
matrix = np.array([
    np.pi, np.pi/4, 0
])

print("Вид кубита на вход:")
print(debug(matrix))

print("Выход после квантовой телепортации:")
print(quantum_teleport(matrix))

Вид кубита на вход:
(np.float64(-0.7071067811865472), np.float64(0.0), np.float64(-0.7071067811865475))
Выход после квантовой телепортации:
(np.float64(-0.7071067811865471), np.float64(0.0), np.float64(-0.7071067811865471))
